# Evaluate Face to Hair Mapper

Inspect the lightweight face-to-hair mapper on held-out paired records and preview top asset matches for sample face profiles.

In [1]:
import pandas as pd
from pathlib import Path
import sys

def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'backend').exists() and (candidate / 'notebooks').exists():
            return candidate
    raise RuntimeError('Could not locate project root.')

PROJECT_ROOT = find_project_root()
BACKEND_ROOT = PROJECT_ROOT / 'backend'
if str(BACKEND_ROOT) not in sys.path:
    sys.path.append(str(BACKEND_ROOT))

from systems.static_auto_tryon.auto_app.ml.datasets import read_jsonl_manifest
from systems.static_auto_tryon.auto_app.ml.face_to_hair_mapper import (
    evaluate_mapper_predictions,
    load_mapper_payload,
    recommend_assets_for_face,
)

PROJECT_ROOT

WindowsPath('D:/Projects/Personal Projects/Hairstyle Recommender Live Tryon')

In [2]:
DATASET_ROOT = PROJECT_ROOT / 'backend' / 'data' / 'datasets' / 'face_to_hair_mapper_light'
VAL_MANIFEST = DATASET_ROOT / 'val.jsonl'
ASSET_BANK_PATH = PROJECT_ROOT / 'backend' / 'data' / 'processed' / 'celeba_hair_rich_assets' / 'reviewed' / 'kept_assets_labeled_enriched.jsonl'
CHECKPOINT_PATH = PROJECT_ROOT / 'backend' / 'models' / 'face_to_hair_mapper_light' / 'mapper_config.json'

CHECKPOINT_PATH

WindowsPath('D:/Projects/Personal Projects/Hairstyle Recommender Live Tryon/backend/models/face_to_hair_mapper_light/mapper_config.json')

In [3]:
val_records = read_jsonl_manifest(VAL_MANIFEST)
asset_rows = read_jsonl_manifest(ASSET_BANK_PATH)
mapper_payload = load_mapper_payload(CHECKPOINT_PATH)
metrics = evaluate_mapper_predictions(val_records, mapper_payload)

pd.Series({
    'record_count': metrics['record_count'],
    'exact_match_accuracy': metrics['exact_match_accuracy'],
    **metrics['field_accuracy'],
})

record_count            15.000000
exact_match_accuracy     0.533333
length                   0.933333
curl                     0.800000
style_family             0.533333
dtype: float64

In [4]:
prediction_df = pd.DataFrame(metrics['prediction_rows'])
prediction_df.head(15)

,asset_id,source_id,true_length,pred_length,correct_length,true_curl,pred_curl,correct_curl,true_style_family,pred_style_family,correct_style_family
0,celeba_hair_000120,000211.jpg,long,long,True,wavy,wavy,True,long_layered,long_layered,True
1,celeba_hair_000246,000426.jpg,long,long,True,wavy,wavy,True,long_layered,long_layered,True
2,celeba_hair_000213,000359.jpg,long,long,True,straight,wavy,False,bob,long_layered,False
3,celeba_hair_000014,000022.jpg,long,long,True,wavy,wavy,True,long_layered,long_layered,True
4,celeba_hair_000003,000007.jpg,short,short,True,straight,straight,True,pompadour,pompadour,True
5,celeba_hair_000009,000015.jpg,short,short,True,straight,straight,True,pompadour,pompadour,True
6,celeba_hair_000185,000318.jpg,short,short,True,straight,straight,True,pompadour,pompadour,True
7,celeba_hair_000029,000043.jpg,long,long,True,wavy,wavy,True,center_part,long_layered,False
8,celeba_hair_000037,000055.jpg,short,short,True,wavy,straight,False,side_part,pompadour,False
9,celeba_hair_000062,000096.jpg,medium,long,False,wavy,wavy,True,bob,long_layered,False


In [5]:
sample_face = val_records[0]['face_labels']
sample_face

{'gender': 'female',
 'face_fullness': 'slim',
 'cheekbones': 'soft',
 'hairline': 'regular'}

In [6]:
recommendation_rows = recommend_assets_for_face(sample_face, asset_rows, mapper_payload, top_k=10)
pd.DataFrame(recommendation_rows)

,asset_id,source_id,image_path,score,hair_labels,gender_label
0,celeba_hair_000002,000006.jpg,D:\Projects\Personal Projects\Hairstyle Recomm...,-13.624753,"{'length': 'long', 'curl': 'wavy', 'style_fami...",female
1,celeba_hair_000011,000018.jpg,D:\Projects\Personal Projects\Hairstyle Recomm...,-13.624753,"{'length': 'long', 'curl': 'wavy', 'style_fami...",female
2,celeba_hair_000012,000019.jpg,D:\Projects\Personal Projects\Hairstyle Recomm...,-13.624753,"{'length': 'long', 'curl': 'wavy', 'style_fami...",female
3,celeba_hair_000014,000022.jpg,D:\Projects\Personal Projects\Hairstyle Recomm...,-13.624753,"{'length': 'long', 'curl': 'wavy', 'style_fami...",female
4,celeba_hair_000025,000034.jpg,D:\Projects\Personal Projects\Hairstyle Recomm...,-13.624753,"{'length': 'long', 'curl': 'wavy', 'style_fami...",female
5,celeba_hair_000102,000178.jpg,D:\Projects\Personal Projects\Hairstyle Recomm...,-13.624753,"{'length': 'long', 'curl': 'wavy', 'style_fami...",female
6,celeba_hair_000120,000211.jpg,D:\Projects\Personal Projects\Hairstyle Recomm...,-13.624753,"{'length': 'long', 'curl': 'wavy', 'style_fami...",female
7,celeba_hair_000133,000232.jpg,D:\Projects\Personal Projects\Hairstyle Recomm...,-13.624753,"{'length': 'long', 'curl': 'wavy', 'style_fami...",female
8,celeba_hair_000148,000257.jpg,D:\Projects\Personal Projects\Hairstyle Recomm...,-13.624753,"{'length': 'long', 'curl': 'wavy', 'style_fami...",female
9,celeba_hair_000149,000259.jpg,D:\Projects\Personal Projects\Hairstyle Recomm...,-13.624753,"{'length': 'long', 'curl': 'wavy', 'style_fami...",female
